# MedGuard AI - Model Training (Logistic Regression Baseline)

In this notebook, we prepare our preprocessed and engineered dataset for machine learning. We will load the data, inspect its dimensions and features, separate the input variables from the target variable, and perform an 80/20 train-test split.

Finally, we will train our baseline machine learning classification model: **Logistic Regression**, and use it to generate class predictions (`y_pred`) on our unseen test set (`X_test`).

### Step 1: Load the Engineered Dataset
We import the necessary libraries and load the cleaned, numeric dataset from our `data/processed` directory.

In [ ]:
# Import pandas library for handling tabular data structures and file operations
import pandas as pd

# Import the train_test_split utility function from scikit-learn for splitting datasets
from sklearn.model_selection import train_test_split

# Load the engineered CSV dataset from the processed folder into a pandas DataFrame object named 'df'
df = pd.read_csv('../data/processed/engineered_appointments.csv')

### Step 2: Display Dataset Shape, Columns, and First Five Rows
We inspect the structure of our DataFrame to verify that it loaded correctly and contains all expected features.

In [ ]:
# Print the shape of the DataFrame (returns a tuple showing the number of rows and total columns)
print(f"Dataset Shape: {df.shape} (Rows, Columns)")

# Convert the DataFrame column names index into a Python list and print them out to inspect our features
print("\nDataset Columns:")
print(df.columns.tolist())

# Display the first 5 rows of the DataFrame inside the notebook using Jupyter's display function
display(df.head())

### Step 3 & 4: Identify the Target Column and Separate Features (X) vs. Target (y)
To train a supervised machine learning model, we must explicitly divide our data into:
*   **`X` (Feature Matrix):** All the variables our model is allowed to use to find patterns.
*   **`y` (Target Vector):** The true outcome variable we want the model to predict (`No_Show`).

In [ ]:
# Extract the target variable column ('No_Show') and assign it to our target vector 'y'
y = df['No_Show']

# Create the feature matrix 'X' by dropping the target column ('No_Show') from the DataFrame
X = df.drop(columns=['No_Show'])

# Print out the shape of the feature matrix X to confirm the target column has been removed
print(f"Features (X) shape: {X.shape}")

# Print out the shape of the target vector y to confirm it matches the number of rows in X
print(f"Target (y) shape: {y.shape}")

### Step 5 & 6: Split the Dataset into 80% Training and 20% Testing
We split the data into a training set (`X_train`, `y_train`) where our model will discover underlying patterns, and a testing set (`X_test`, `y_test`) that acts as an unseen final exam to evaluate the model's true predictive power.

*Note: We explicitly use `stratify=y` so that the ratio of Shows vs. No-Shows remains perfectly balanced across both splits, preventing any class imbalance bias.*

In [ ]:
# Split the features and target arrays into random train and test subsets
# X: Feature matrix to be split
# y: Target vector to be split
# test_size=0.20: Allocates exactly 20% of the data for testing and 80% for training
# random_state=42: Sets a fixed random seed so that the exact same split occurs every time this code is run
# stratify=y: Ensures both training and testing subsets contain the exact same class proportions as the original dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# Print a confirmation message indicating the split operation succeeded
print("Dataset successfully split into 80% Training and 20% Testing subsets!\n")

# Print the exact dimensions (rows and columns) of the training feature matrix
print(f"Training features (X_train) shape: {X_train.shape}")

# Print the exact length (number of rows) of the training target vector
print(f"Training target (y_train) shape:   {y_train.shape}")

# Print the exact dimensions (rows and columns) of the testing feature matrix
print(f"Testing features (X_test) shape:   {X_test.shape}")

# Print the exact length (number of rows) of the testing target vector
print(f"Testing target (y_test) shape:     {y_test.shape}")

### Step 7: Initialize and Train the Logistic Regression Model
Logistic Regression is a foundational, interpretable linear classification algorithm. It computes a weighted linear combination of the input features (`WaitingDays`, `Age`, `SMS_received`, etc.) and passes the output through a sigmoid (logistic) curve to predict the exact probability (0.0 to 1.0) that a patient will be a `No-Show` (`1`).

In [ ]:
# Import LogisticRegression class from scikit-learn's linear_model module
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression classifier
# max_iter=1000: Increases the maximum number of optimization steps to ensure the solver converges smoothly across many features
# random_state=42: Sets a fixed random seed for reproducible training across runs
model = LogisticRegression(max_iter=1000, random_state=42)

# Train (fit) the Logistic Regression model using only our 80% training feature matrix and target vector
model.fit(X_train, y_train)

# Print a confirmation message indicating that model training successfully finished
print("Logistic Regression model successfully trained on X_train and y_train!")

### Step 8: Generate Predictions on the Test Set (`X_test`)
Now that the model has learned historical patterns from `X_train`, we use it to predict binary class labels (`0` for Show, `1` for No-Show) for our held-out test features (`X_test`) and store them in `y_pred`.

In [ ]:
# Use the trained Logistic Regression model to predict binary class labels (0 or 1) for the unseen testing data X_test and store them in y_pred
y_pred = model.predict(X_test)

# Print the exact confirmation message saying predictions completed successfully
print("Predictions completed successfully!")

### Step 9: Baseline Model Evaluation
We evaluate the Logistic Regression model using standard classification metrics.

In [ ]:
# Import required metrics from scikit-learn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Calculate the overall accuracy of the predictions on the test set
accuracy = accuracy_score(y_test, y_pred)

# Calculate the precision (true positives / (true positives + false positives)) to see how many predicted No-Shows were actual No-Shows
precision = precision_score(y_test, y_pred, zero_division=0)

# Calculate the recall (true positives / (true positives + false negatives)) to see what percentage of actual No-Shows the model caught
recall = recall_score(y_test, y_pred, zero_division=0)

# Calculate the F1-score, which is the harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred, zero_division=0)

# Generate the confusion matrix to see the exact counts of true/false positives and negatives
conf_matrix = confusion_matrix(y_test, y_pred)

# Print the calculated metrics clearly, formatted to 4 decimal places
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Print the confusion matrix clearly
print("\nConfusion Matrix:")
print(conf_matrix)

### Step 10: Class Imbalance Analysis
We investigate the distribution of our target variable (`No_Show`) in the training data to understand why our model is struggling with recall.

In [ ]:
# Import matplotlib for plotting the bar chart
import matplotlib.pyplot as plt

# Calculate the exact number of Show (0) cases in the training set
show_count = (y_train == 0).sum()

# Calculate the exact number of No-show (1) cases in the training set
noshow_count = (y_train == 1).sum()

# Get the total number of training records to calculate percentages
total_count = len(y_train)

# Calculate the percentage of Show cases
show_pct = (show_count / total_count) * 100

# Calculate the percentage of No-show cases
noshow_pct = (noshow_count / total_count) * 100

# Identify which class is the majority and which is the minority
majority_class = 'Show (0)' if show_count > noshow_count else 'No-show (1)'
minority_class = 'No-show (1)' if show_count > noshow_count else 'Show (0)'

# Calculate the imbalance ratio (Majority Count / Minority Count)
majority_count = max(show_count, noshow_count)
minority_count = min(show_count, noshow_count)
imbalance_ratio = majority_count / minority_count

# Print out the calculated statistics clearly
print(f"Show (0) count: {show_count} ({show_pct:.2f}%)")
print(f"No-show (1) count: {noshow_count} ({noshow_pct:.2f}%)")
print(f"Majority Class: {majority_class}")
print(f"Minority Class: {minority_class}")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}:1")

# Create a simple bar chart to visualize the class distribution
plt.figure(figsize=(6, 4))
plt.bar(['Show (0)', 'No-show (1)'], [show_count, noshow_count], color=['blue', 'red'])
plt.title('Target Class Distribution in Training Data')
plt.ylabel('Number of Patients')
plt.show()

### Step 11: Train a Class-Weighted Logistic Regression Model
To combat the ~4:1 class imbalance, we will train a second Logistic Regression model. This time, we will use the `class_weight="balanced"` parameter. This forces the model to heavily penalize errors made on the minority class (No-shows), effectively forcing it to pay 4 times more attention to catching No-shows rather than just taking the easy route of predicting everyone will show up.

In [ ]:
# Initialize a NEW Logistic Regression model specifically for the class-weighting experiment
# max_iter=1000 ensures the solver has enough steps to converge
# random_state=42 ensures reproducibility
# class_weight='balanced' automatically adjusts weights inversely proportional to class frequencies (penalizing minority misclassifications 4x more)
weighted_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

# Train (fit) the new weighted model using our existing 80% training feature matrix and target vector
weighted_model.fit(X_train, y_train)

# Use the trained weighted model to predict binary class labels (0 or 1) for the unseen testing data X_test
# We store these predictions in a NEW variable 'weighted_y_pred' so we don't overwrite our baseline predictions
weighted_y_pred = weighted_model.predict(X_test)

# Print a clear confirmation message indicating that model training and prediction successfully finished
print("Class-weighted Logistic Regression model successfully trained and predictions generated!")

### Step 12: Evaluate the Class-Weighted Logistic Regression Model
We evaluate the new `weighted_model` using the exact same metrics as our baseline to fairly compare how the `class_weight='balanced'` parameter impacted performance, particularly Recall.

In [ ]:
# Calculate the overall accuracy using the true labels (y_test) and the new weighted predictions (weighted_y_pred)
accuracy_w = accuracy_score(y_test, weighted_y_pred)

# Calculate the precision for the weighted model
precision_w = precision_score(y_test, weighted_y_pred, zero_division=0)

# Calculate the recall for the weighted model (This is the critical metric we want to improve!)
recall_w = recall_score(y_test, weighted_y_pred, zero_division=0)

# Calculate the F1-score for the weighted model
f1_w = f1_score(y_test, weighted_y_pred, zero_division=0)

# Generate the new confusion matrix to see how the True/False Positives and Negatives have shifted
conf_matrix_w = confusion_matrix(y_test, weighted_y_pred)

# Print the calculated metrics clearly, formatted to 4 decimal places
print("=" * 50)
print("CLASS-WEIGHTED LOGISTIC REGRESSION EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_w:.4f}")
print(f"Precision: {precision_w:.4f}")
print(f"Recall:    {recall_w:.4f}")
print(f"F1 Score:  {f1_w:.4f}")

# Print the new confusion matrix clearly
print("\nConfusion Matrix:")
print(conf_matrix_w)
print("=" * 50)


### Step 13: ROC-AUC Evaluation
To better understand the overall diagnostic ability of our class-weighted model across all possible classification thresholds (not just the default 50% cutoff), we evaluate its ROC-AUC (Receiver Operating Characteristic - Area Under the Curve) score. This metric tells us how well the model separates the 'Show' and 'No-show' classes regardless of the specific probability threshold chosen.

In [ ]:
# Import the roc_auc_score function from scikit-learn metrics module
from sklearn.metrics import roc_auc_score

# Obtain predicted probabilities for the positive class (Class 1: No-show)
# .predict_proba() returns a 2D array where column 0 is probability of class 0, and column 1 is probability of class 1
weighted_y_prob = weighted_model.predict_proba(X_test)[:, 1]

# Calculate the ROC-AUC score using the true test labels and the continuous probability predictions
roc_auc = roc_auc_score(y_test, weighted_y_prob)

# Print the calculated ROC-AUC score clearly, formatted to 4 decimal places
print("=" * 50)
print("CLASS-WEIGHTED LOGISTIC REGRESSION ROC-AUC")
print("=" * 50)
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("=" * 50)


### Step 14: Threshold Analysis
By default, Logistic Regression uses a probability threshold of 0.50 to decide between Show (0) and No-show (1). Because we care significantly about Recall (catching high-risk patients), we might want to lower or raise this threshold. Here, we analyze how different thresholds impact our evaluation metrics.

In [ ]:
# Define a list of probability thresholds we want to evaluate
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

# Create an empty list to store the results dictionary for each threshold
results = []

# Loop through each threshold value
for t in thresholds:
    # Create a temporary prediction array: if the probability is >= threshold, predict 1 (No-show), else 0 (Show)
    temp_y_pred = (weighted_y_prob >= t).astype(int)
    
    # Calculate the exact metrics for this specific threshold
    acc = accuracy_score(y_test, temp_y_pred)
    prec = precision_score(y_test, temp_y_pred, zero_division=0)
    rec = recall_score(y_test, temp_y_pred, zero_division=0)
    f1 = f1_score(y_test, temp_y_pred, zero_division=0)
    
    # Append a dictionary with the calculated metrics to our results list
    results.append({
        'Threshold': t, 
        'Accuracy': acc, 
        'Precision': prec, 
        'Recall': rec, 
        'F1-Score': f1
    })

# Convert the list of dictionaries into a Pandas DataFrame for clean tabular formatting
df_thresholds = pd.DataFrame(results)

# Print out the final table clearly
print("=" * 60)
print("           THRESHOLD PERFORMANCE ANALYSIS")
print("=" * 60)
print(df_thresholds.to_string(index=False))
print("=" * 60)


### Step 15: Train a Random Forest Baseline Model
To see if a more complex, non-linear model can find deeper patterns in the data, we will train a Random Forest classifier. We continue to use `class_weight="balanced"` to force the model to handle the 4:1 class imbalance.

In [ ]:
# Import the RandomForestClassifier from scikit-learn's ensemble module
from sklearn.ensemble import RandomForestClassifier

# Initialize a NEW Random Forest model
# n_estimators=200: We will use 200 decision trees in our forest for robust predictions
# random_state=42: Set fixed random seed for reproducible results
# class_weight='balanced': Automatically adjust weights to heavily penalize minority class (No-show) errors
random_forest_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')

# Train (fit) the Random Forest model using our existing 80% training features and target
random_forest_model.fit(X_train, y_train)

# Generate hard predictions (0 or 1 labels using the default 0.50 threshold) on the testing data
# Store them in a NEW variable so we do not overwrite our Logistic Regression predictions
random_forest_y_pred = random_forest_model.predict(X_test)

# Generate the predicted probability scores specifically for the positive class (Class 1: No-show)
# This is required for future ROC-AUC calculation and threshold analysis
random_forest_y_prob = random_forest_model.predict_proba(X_test)[:, 1]

# Print a confirmation message indicating that training and prediction successfully finished
print("Random Forest (Class-weighted) successfully trained!")
print("Hard predictions and probability scores successfully generated!")


### Step 16: Random Forest Baseline Evaluation
We evaluate the complex, non-linear Random Forest model using the exact same metrics as our Logistic Regression models to see if it provides a better tradeoff between Precision and Recall. Note: we are evaluating at the default 50% threshold.

In [ ]:
# Calculate the overall accuracy for the Random Forest model
accuracy_rf = accuracy_score(y_test, random_forest_y_pred)

# Calculate the precision for the Random Forest model
precision_rf = precision_score(y_test, random_forest_y_pred, zero_division=0)

# Calculate the recall for the Random Forest model
recall_rf = recall_score(y_test, random_forest_y_pred, zero_division=0)

# Calculate the F1-score for the Random Forest model
f1_rf = f1_score(y_test, random_forest_y_pred, zero_division=0)

# Calculate the ROC-AUC score using the raw probability predictions
roc_auc_rf = roc_auc_score(y_test, random_forest_y_prob)

# Generate the confusion matrix to see the True/False Positives and Negatives
conf_matrix_rf = confusion_matrix(y_test, random_forest_y_pred)

# Print all calculated metrics clearly, formatted to 4 decimal places
print("=" * 50)
print("RANDOM FOREST (CLASS-WEIGHTED) EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall:    {recall_rf:.4f}")
print(f"F1 Score:  {f1_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_rf:.4f}")

# Print the confusion matrix clearly
print("\nConfusion Matrix:")
print(conf_matrix_rf)
print("=" * 50)


### Step 17: Train an XGBoost Baseline Model
XGBoost (eXtreme Gradient Boosting) is a powerful tree-based ensemble method. Since XGBoost does not have a native `class_weight='balanced'` parameter like Scikit-Learn models, we calculate the class imbalance ratio manually from `y_train` and pass it to the `scale_pos_weight` parameter to penalize minority class errors.

In [ ]:
# Import the XGBClassifier from the xgboost library
# Make sure xgboost is installed via: pip install xgboost
from xgboost import XGBClassifier

# Calculate the exact count of majority (Show, 0) and minority (No-show, 1) cases in the training set
num_show = (y_train == 0).sum()
num_noshow = (y_train == 1).sum()

# Calculate the scale_pos_weight by taking the ratio of majority cases to minority cases (~3.95)
scale_pos_weight_value = num_show / num_noshow

# Initialize a NEW XGBoost classifier model with our baseline parameters
# n_estimators=200: Train 200 sequential boosting rounds (trees)
# max_depth=4: Restrict each tree to a depth of 4 to prevent overfitting
# learning_rate=0.05: Shrink feature weights at each step to make learning more robust
# subsample=0.8, colsample_bytree=0.8: Randomly sample 80% of rows and columns per tree to add variance and prevent overfitting
# scale_pos_weight: Hand-calculated class imbalance ratio to penalize No-show errors heavily
# random_state=42: For reproducibility
# eval_metric="logloss": Use log loss to monitor optimization
xgboost_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_value,
    random_state=42,
    eval_metric="logloss"
)

# Train (fit) the XGBoost model using the training features and targets
xgboost_model.fit(X_train, y_train)

# Generate hard predictions (0 or 1 labels using default 0.50 cutoff) and store them
xgboost_y_pred = xgboost_model.predict(X_test)

# Generate probability scores specifically for the positive class (Class 1: No-show)
xgboost_y_prob = xgboost_model.predict_proba(X_test)[:, 1]

# Print a confirmation message indicating successful execution
print("XGBoost Baseline model successfully trained!")
print("Hard predictions and probability scores successfully generated!")


### Step 18: XGBoost Baseline Evaluation
We evaluate the XGBoost baseline model using the exact same metrics as our Logistic Regression and Random Forest models. This allows us to see how the gradient boosting approach compares to the others before any hyperparameter tuning.

In [ ]:
# Calculate the overall accuracy for the XGBoost model
accuracy_xgb = accuracy_score(y_test, xgboost_y_pred)

# Calculate the precision for the XGBoost model
precision_xgb = precision_score(y_test, xgboost_y_pred, zero_division=0)

# Calculate the recall for the XGBoost model
recall_xgb = recall_score(y_test, xgboost_y_pred, zero_division=0)

# Calculate the F1-score for the XGBoost model
f1_xgb = f1_score(y_test, xgboost_y_pred, zero_division=0)

# Calculate the ROC-AUC score using the raw probability predictions
roc_auc_xgb = roc_auc_score(y_test, xgboost_y_prob)

# Generate the confusion matrix to see the True/False Positives and Negatives
conf_matrix_xgb = confusion_matrix(y_test, xgboost_y_pred)

# Print all calculated metrics clearly, formatted to 4 decimal places
print("=" * 50)
print("XGBOOST BASELINE EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall:    {recall_xgb:.4f}")
print(f"F1 Score:  {f1_xgb:.4f}")
print(f"ROC-AUC:   {roc_auc_xgb:.4f}")

# Print the confusion matrix clearly
print("\nConfusion Matrix:")
print(conf_matrix_xgb)
print("=" * 50)


### Step 19: Create a Validation Split
To prepare for hyperparameter tuning and model optimization without overfitting to our test set, we create a strict validation split. We take our existing `X_train` and split it further into a model-training portion (`X_train_model`) and a validation portion (`X_val`).

From this point onward, `X_test` and `y_test` represent a truly untouched FINAL test set.

In [ ]:
# We import train_test_split again for clarity, though it is already imported above
from sklearn.model_selection import train_test_split

# Split the existing training data (80% of original) into a new training set and a validation set
# test_size=0.20: Allocate 20% of the training data strictly for validation
# random_state=42: Fixed seed for reproducibility
# stratify=y_train: Ensure the ~4:1 class imbalance is perfectly maintained in both new subsets
X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train, 
    y_train, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_train
)

# Print the exact dimensions of all three major dataset splits
print("============================================================")
print("                   DATASET SHAPES                           ")
print("============================================================")
print(f"X_train_model shape: {X_train_model.shape}")
print(f"y_train_model shape: {y_train_model.shape}")
print("-" * 60)
print(f"X_val shape:         {X_val.shape}")
print(f"y_val shape:         {y_val.shape}")
print("-" * 60)
print(f"X_test shape:        {X_test.shape}   <-- FINAL TEST SET (UNTOUCHED)")
print(f"y_test shape:        {y_test.shape}   <-- FINAL TEST SET (UNTOUCHED)")
print("============================================================\n")

# Calculate and print the class distributions for the new subsets to verify stratification
print("============================================================")
print("                   CLASS DISTRIBUTIONS                      ")
print("============================================================")
print("y_train_model class counts:")
print(y_train_model.value_counts().to_string())
print("\ny_val class counts:")
print(y_val.value_counts().to_string())
print("============================================================")


### Step 20: Retrain the XGBoost Baseline Using the New Training Portion
To validate future tuning properly without touching `X_test`, we first retrain our XGBoost baseline using strictly the `X_train_model` portion and test it on our newly created `X_val` validation set.

In [ ]:
# Calculate the exact count of majority (Show, 0) and minority (No-show, 1) cases ONLY in the new model training subset
num_show_val = (y_train_model == 0).sum()
num_noshow_val = (y_train_model == 1).sum()

# Calculate the scale_pos_weight for the validation split experiment
scale_pos_weight_val = num_show_val / num_noshow_val

# Initialize a NEW XGBoost classifier specifically for validation-set testing
# n_estimators=200: Train 200 sequential boosting rounds
# max_depth=4: Restrict tree depth to 4
# learning_rate=0.05: Shrink weights at each step
# subsample=0.8, colsample_bytree=0.8: Random sampling to prevent overfitting
# scale_pos_weight: Hand-calculated class imbalance ratio from y_train_model
# random_state=42: Fixed seed for reproducibility
# eval_metric="logloss": Use log loss to monitor optimization
xgboost_validation_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_val,
    random_state=42,
    eval_metric="logloss"
)

# Train (fit) the XGBoost model using ONLY the X_train_model and y_train_model subsets
xgboost_validation_model.fit(X_train_model, y_train_model)

# Generate hard predictions (0 or 1 labels) strictly on the newly created validation set X_val
xgboost_val_pred = xgboost_validation_model.predict(X_val)

# Generate predicted probabilities for the positive class strictly on X_val
xgboost_val_prob = xgboost_validation_model.predict_proba(X_val)[:, 1]

# Print a clear confirmation message indicating successful execution
print("XGBoost Validation model successfully trained on X_train_model!")
print("Hard predictions and probability scores successfully generated for X_val!")


### Step 21: Evaluate the XGBoost Validation Baseline
We evaluate the XGBoost validation model strictly on the new `y_val` targets using the same evaluation metrics as before. This establishes the baseline performance on the validation set, which we will aim to beat during hyperparameter tuning.

In [ ]:
# Calculate the overall accuracy on the validation set
accuracy_val = accuracy_score(y_val, xgboost_val_pred)

# Calculate the precision on the validation set
precision_val = precision_score(y_val, xgboost_val_pred, zero_division=0)

# Calculate the recall on the validation set
recall_val = recall_score(y_val, xgboost_val_pred, zero_division=0)

# Calculate the F1-score on the validation set
f1_val = f1_score(y_val, xgboost_val_pred, zero_division=0)

# Calculate the ROC-AUC score on the validation set
roc_auc_val = roc_auc_score(y_val, xgboost_val_prob)

# Generate the confusion matrix for the validation set
conf_matrix_val = confusion_matrix(y_val, xgboost_val_pred)

# Print all calculated metrics clearly, formatted to 4 decimal places
print("=" * 50)
print("XGBOOST VALIDATION BASELINE EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_val:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall:    {recall_val:.4f}")
print(f"F1 Score:  {f1_val:.4f}")
print(f"ROC-AUC:   {roc_auc_val:.4f}")

# Print the confusion matrix clearly
print("\nConfusion Matrix:")
print(conf_matrix_val)
print("=" * 50)


### Step 22: XGBoost Hyperparameter Tuning
Now we perform a controlled hyperparameter search to find the best configuration for XGBoost. We iterate through a predefined grid of `n_estimators`, `max_depth`, and `learning_rate` while keeping other parameters fixed. We strictly use `X_train_model` for training and `X_val` for evaluating the ROC-AUC score to prevent any data leakage from the final `X_test` set.

In [ ]:
# Import itertools to easily create combinations of hyperparameters
import itertools

# Define the parameter grid to search over as requested
n_estimators_list = [100, 200, 300]
max_depth_list = [3, 4, 5]
learning_rate_list = [0.03, 0.05, 0.1]

# Create an empty list to store the results of each combination
tuning_results = []

# Calculate scale_pos_weight specifically from the model training subset
num_show_train_model = (y_train_model == 0).sum()
num_noshow_train_model = (y_train_model == 1).sum()
scale_pos_weight_tune = num_show_train_model / num_noshow_train_model

# Generate all possible combinations of the three hyperparameters
param_combinations = list(itertools.product(n_estimators_list, max_depth_list, learning_rate_list))
print(f"Total configurations to test: {len(param_combinations)}\n")

# Loop through each hyperparameter combination
for n_est, m_depth, lr in param_combinations:
    
    # Initialize a temporary XGBoost model with the specific parameters for this loop iteration
    temp_model = XGBClassifier(
        n_estimators=n_est,
        max_depth=m_depth,
        learning_rate=lr,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_tune,
        random_state=42,
        eval_metric="logloss"
    )
    
    # Train the temporary model using ONLY the model training subset
    temp_model.fit(X_train_model, y_train_model)
    
    # Generate predicted probabilities strictly on the validation set X_val
    temp_val_prob = temp_model.predict_proba(X_val)[:, 1]
    
    # Calculate the ROC-AUC score for the validation set
    temp_roc_auc = roc_auc_score(y_val, temp_val_prob)
    
    # Append the parameters and the resulting validation ROC-AUC to our results list
    tuning_results.append({
        'n_estimators': n_est,
        'max_depth': m_depth,
        'learning_rate': lr,
        'Val_ROC_AUC': temp_roc_auc
    })

# Convert the list of results into a Pandas DataFrame for easy viewing and sorting
df_tuning_results = pd.DataFrame(tuning_results)

# Sort the results in descending order by Validation ROC-AUC so the best model is at the top
df_tuning_results = df_tuning_results.sort_values(by='Val_ROC_AUC', ascending=False).reset_index(drop=True)

# Extract the best parameter combination (the first row after sorting)
best_xgb_params = df_tuning_results.iloc[0].to_dict()

# Print out the full sorted results table clearly
print("============================================================")
print("             XGBOOST HYPERPARAMETER TUNING RESULTS          ")
print("============================================================")
print(df_tuning_results.to_string())
print("============================================================\n")

# Print the best parameter combination and its corresponding score
print("============================================================")
print("             BEST PARAMETER CONFIGURATION                   ")
print("============================================================")
print(f"Best Parameters: {best_xgb_params}")
print(f"Best Validation ROC-AUC: {best_xgb_params['Val_ROC_AUC']:.4f}")
print("============================================================")


### Step 23: Validation Threshold Analysis for the Tuned XGBoost Model
We now train our final optimally-tuned XGBoost model (using `X_train_model`) and evaluate how different classification probability thresholds affect performance on `X_val`. This will help us identify the optimal cutoff before applying anything to our untouched test set.

In [ ]:
# Calculate scale_pos_weight from the model training subset again for clarity
num_show_train_model = (y_train_model == 0).sum()
num_noshow_train_model = (y_train_model == 1).sum()
scale_pos_weight_tuned = num_show_train_model / num_noshow_train_model

# Initialize the NEW XGBoost model with the BEST hyperparameters found in Step 22
tuned_xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_tuned,
    random_state=42,
    eval_metric="logloss"
)

# Train the tuned model using strictly the model training subset
tuned_xgboost_model.fit(X_train_model, y_train_model)

# Generate probability scores for the No-show class specifically on the validation set X_val
tuned_xgb_val_prob = tuned_xgboost_model.predict_proba(X_val)[:, 1]

# Define the exhaustive list of classification thresholds to test
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

# Initialize an empty list to collect the evaluation metrics for each threshold
thresh_results = []

# Loop through every threshold
for t in thresholds:
    # Convert probabilities to binary predictions based on the current threshold
    temp_y_pred = (tuned_xgb_val_prob >= t).astype(int)
    
    # Calculate all required evaluation metrics against the true validation labels (y_val)
    acc = accuracy_score(y_val, temp_y_pred)
    prec = precision_score(y_val, temp_y_pred, zero_division=0)
    rec = recall_score(y_val, temp_y_pred, zero_division=0)
    f1 = f1_score(y_val, temp_y_pred, zero_division=0)
    
    # Store the threshold and its corresponding metrics in the list
    thresh_results.append({
        'Threshold': t,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

# Convert the list of results into a pandas DataFrame for structured display
df_val_thresholds = pd.DataFrame(thresh_results)

# Print the complete threshold analysis table clearly
print("================================================================")
print("     TUNED XGBOOST VALIDATION THRESHOLD ANALYSIS                ")
print("================================================================")
print(df_val_thresholds.to_string(index=False))
print("================================================================\n")

# Identify the row with the absolute highest F1-score
best_f1_row = df_val_thresholds.loc[df_val_thresholds['F1-Score'].idxmax()]

# Store the highest-F1 threshold and its corresponding metrics separately
best_f1_threshold = best_f1_row['Threshold']
best_f1_acc = best_f1_row['Accuracy']
best_f1_prec = best_f1_row['Precision']
best_f1_rec = best_f1_row['Recall']
best_f1_score = best_f1_row['F1-Score']

print("================================================================")
print("             BEST THRESHOLDS IDENTIFIED ON VALIDATION SET       ")
print("================================================================")
print(f"Optimal F1 Threshold: {best_f1_threshold:.2f}")
print(f"  - Accuracy:  {best_f1_acc:.4f}")
print(f"  - Precision: {best_f1_prec:.4f}")
print(f"  - Recall:    {best_f1_rec:.4f}")
print(f"  - F1-Score:  {best_f1_score:.4f}\n")

# Identify the threshold that gives the highest recall while maintaining Precision >= 0.30
# First, filter the dataframe to only include rows where Precision >= 0.30
high_prec_df = df_val_thresholds[df_val_thresholds['Precision'] >= 0.30]

# Check if any such threshold exists
if not high_prec_df.empty:
    # If exists, find the row with the maximum recall among those valid rows
    best_rec_row = high_prec_df.loc[high_prec_df['Recall'].idxmax()]
    print(f"Threshold for Max Recall (Precision >= 0.30): {best_rec_row['Threshold']:.2f}")
    print(f"  - Accuracy:  {best_rec_row['Accuracy']:.4f}")
    print(f"  - Precision: {best_rec_row['Precision']:.4f}")
    print(f"  - Recall:    {best_rec_row['Recall']:.4f}")
    print(f"  - F1-Score:  {best_rec_row['F1-Score']:.4f}")
else:
    print("No threshold found maintaining Precision >= 0.30.")
print("================================================================")


### Detailed Validation Threshold Analysis
Because we observed a low Accuracy and Precision at the 0.50 threshold, we now analyze a more granular range of thresholds (0.45 to 0.70 in 0.01 increments). This allows us to find a specific threshold that might provide a better tradeoff between Accuracy, Precision, and Recall before finalizing our decision.

In [ ]:
# Import numpy for generating the fine-grained threshold range
import numpy as np

# Create an array of thresholds from 0.45 to 0.70 in increments of 0.01
fine_thresholds = np.arange(0.45, 0.71, 0.01)

# Initialize an empty list to store the detailed results
fine_results = []

# Loop through each fine-grained threshold
for t in fine_thresholds:
    # Convert validation probabilities to binary predictions based on the current threshold
    temp_y_pred = (tuned_xgb_val_prob >= t).astype(int)
    
    # Calculate core evaluation metrics
    acc = accuracy_score(y_val, temp_y_pred)
    prec = precision_score(y_val, temp_y_pred, zero_division=0)
    rec = recall_score(y_val, temp_y_pred, zero_division=0)
    f1 = f1_score(y_val, temp_y_pred, zero_division=0)
    
    # Calculate Confusion Matrix elements (True Negatives, False Positives, False Negatives, True Positives)
    tn, fp, fn, tp = confusion_matrix(y_val, temp_y_pred).ravel()
    
    # Append the results, rounding the threshold for clean display
    fine_results.append({
        'Threshold': round(t, 2),
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn
    })

# Convert the list into a pandas DataFrame
df_fine = pd.DataFrame(fine_results)

# -- IDENTIFY CANDIDATE THRESHOLDS --

# 1. Threshold with the highest F1-score
best_f1_idx = df_fine['F1-Score'].idxmax()
best_f1_row = df_fine.loc[best_f1_idx]

# 2. Highest precision threshold with Recall >= 0.70
rec_70_df = df_fine[df_fine['Recall'] >= 0.70]
if not rec_70_df.empty:
    best_prec_rec70_idx = rec_70_df['Precision'].idxmax()
    best_prec_rec70_row = rec_70_df.loc[best_prec_rec70_idx]
else:
    best_prec_rec70_row = None

# 3. Highest precision threshold with Recall >= 0.60
rec_60_df = df_fine[df_fine['Recall'] >= 0.60]
if not rec_60_df.empty:
    best_prec_rec60_idx = rec_60_df['Precision'].idxmax()
    best_prec_rec60_row = rec_60_df.loc[best_prec_rec60_idx]
else:
    best_prec_rec60_row = None

# 4. Highest precision threshold with Recall >= 0.50
rec_50_df = df_fine[df_fine['Recall'] >= 0.50]
if not rec_50_df.empty:
    best_prec_rec50_idx = rec_50_df['Precision'].idxmax()
    best_prec_rec50_row = rec_50_df.loc[best_prec_rec50_idx]
else:
    best_prec_rec50_row = None

# 5. Threshold with Accuracy closest to 0.70 while maintaining Recall >= 0.50
if not rec_50_df.empty:
    # Calculate absolute difference from 0.70 for Accuracy
    acc_diffs = (rec_50_df['Accuracy'] - 0.70).abs()
    closest_acc_idx = acc_diffs.idxmin()
    closest_acc_row = rec_50_df.loc[closest_acc_idx]
else:
    closest_acc_row = None

# Print the complete detailed threshold table clearly
print("================================================================================")
print("             DETAILED TUNED XGBOOST THRESHOLD ANALYSIS (0.45 - 0.70)            ")
print("================================================================================")
print(df_fine.to_string(index=False))
print("================================================================================\n")

# Print the specifically requested candidate thresholds
print("================================================================================")
print("             CANDIDATE THRESHOLDS BASED ON SPECIFIC CRITERIA                    ")
print("================================================================================")
print("1. Highest F1-Score:")
print(f"   Threshold: {best_f1_row['Threshold']:.2f} | F1: {best_f1_row['F1-Score']:.4f} | Acc: {best_f1_row['Accuracy']:.4f} | Prec: {best_f1_row['Precision']:.4f} | Rec: {best_f1_row['Recall']:.4f}")

if best_prec_rec70_row is not None:
    print("\n2. Highest Precision while Recall >= 0.70:")
    print(f"   Threshold: {best_prec_rec70_row['Threshold']:.2f} | Prec: {best_prec_rec70_row['Precision']:.4f} | Rec: {best_prec_rec70_row['Recall']:.4f} | Acc: {best_prec_rec70_row['Accuracy']:.4f}")

if best_prec_rec60_row is not None:
    print("\n3. Highest Precision while Recall >= 0.60:")
    print(f"   Threshold: {best_prec_rec60_row['Threshold']:.2f} | Prec: {best_prec_rec60_row['Precision']:.4f} | Rec: {best_prec_rec60_row['Recall']:.4f} | Acc: {best_prec_rec60_row['Accuracy']:.4f}")

if best_prec_rec50_row is not None:
    print("\n4. Highest Precision while Recall >= 0.50:")
    print(f"   Threshold: {best_prec_rec50_row['Threshold']:.2f} | Prec: {best_prec_rec50_row['Precision']:.4f} | Rec: {best_prec_rec50_row['Recall']:.4f} | Acc: {best_prec_rec50_row['Accuracy']:.4f}")

if closest_acc_row is not None:
    print("\n5. Accuracy closest to 0.70 while Recall >= 0.50:")
    print(f"   Threshold: {closest_acc_row['Threshold']:.2f} | Acc: {closest_acc_row['Accuracy']:.4f} | Prec: {closest_acc_row['Precision']:.4f} | Rec: {closest_acc_row['Recall']:.4f}")
print("================================================================================")


### Step 24: Lock the Final Model Configuration
After extensive tuning and threshold analysis, we have selected our final model configuration. We choose the `0.55` threshold because it balances the need to catch high-risk patients (Recall > 70%) while reducing false alarms (improving Precision and Accuracy compared to `0.50`), maintaining a strong overall F1-score.

In [ ]:
# Create a dictionary to rigidly store the final locked-in model configuration and its expected performance
final_model_config = {
    # Store the chosen algorithm name
    "Model_Type": "XGBoost",
    
    # Store the exact hyperparameter values discovered during tuning (Step 22)
    "Hyperparameters": {
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "eval_metric": "logloss"
    },
    
    # Store the class imbalance handling strategy used
    "Class_Imbalance_Strategy": "scale_pos_weight calculated from y_train_model",
    
    # Store the final selected classification probability threshold (Step 23)
    "Operating_Threshold": 0.55,
    
    # Store the validation performance metrics achieved exactly at this threshold
    "Validation_Performance": {
        "Accuracy": 0.647704,
        "Precision": 0.327407,
        "Recall": 0.706723,
        "F1-Score": 0.447499,
        "ROC-AUC": 0.7397
    },
    
    # Store a clear explanation detailing exactly why this specific threshold was chosen for MedGuard AI
    "Threshold_Rationale": (
        "Threshold 0.55 was selected as the operating threshold because it maintains "
        "Recall above 70% while improving Precision and Accuracy compared with "
        "threshold 0.50, with essentially unchanged F1. This provides a useful balance "
        "between catching no-shows and avoiding excessive false alarms for MedGuard AI."
    )
}

# Print a formatted output header to clearly demarcate the final configuration
print("================================================================================")
print("                    FINAL MODEL CONFIGURATION LOCKED                            ")
print("================================================================================")

# Iterate through the dictionary and cleanly print each configuration section
for key, value in final_model_config.items():
    # If the value is a nested dictionary (like Hyperparameters or Validation_Performance)
    if isinstance(value, dict):
        # Print the section header, replacing underscores with spaces for readability
        print(f"\n{key.replace('_', ' ')}:")
        # Loop through and print the inner key-value pairs
        for sub_key, sub_value in value.items():
            print(f"  - {sub_key}: {sub_value}")
    else:
        # If the value is a single string or number, print it directly under its header
        print(f"\n{key.replace('_', ' ')}:")
        print(f"  {value}")
        
# Print a final divider
print("\n================================================================================")


### Step 25: Final Test Set Evaluation
In this final step, we train our chosen model configuration on the full 80% training dataset (`X_train`, `y_train`) and evaluate it exactly ONCE on the completely unseen 20% test dataset (`X_test`, `y_test`) using our locked operating threshold of `0.55`. This provides the true, unbiased estimate of how MedGuard AI will perform in the real world.

In [ ]:
# 1. Calculate the final scale_pos_weight using the FULL original training set (y_train)
num_show_full_train = (y_train == 0).sum()
num_noshow_full_train = (y_train == 1).sum()
scale_pos_weight_final = num_show_full_train / num_noshow_full_train

# 2. Initialize the final XGBoost model with our rigidly locked hyperparameters
final_xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_final,
    random_state=42,
    eval_metric="logloss"
)

# 3. Train the model on the full original training dataset (X_train, y_train)
final_xgboost_model.fit(X_train, y_train)

# 4. Generate positive-class (No-show) probabilities strictly on the UNTOUCHED test set (X_test)
final_test_prob = final_xgboost_model.predict_proba(X_test)[:, 1]

# 5. Apply our locked operating threshold of 0.55 to generate hard binary predictions
locked_threshold = 0.55
final_test_pred = (final_test_prob >= locked_threshold).astype(int)

# 6. Calculate all final evaluation metrics against the unseen true test labels (y_test)
final_accuracy = accuracy_score(y_test, final_test_pred)
final_precision = precision_score(y_test, final_test_pred, zero_division=0)
final_recall = recall_score(y_test, final_test_pred, zero_division=0)
final_f1 = f1_score(y_test, final_test_pred, zero_division=0)
final_roc_auc = roc_auc_score(y_test, final_test_prob)

# 7. Extract Confusion Matrix elements to see exact prediction counts
tn_final, fp_final, fn_final, tp_final = confusion_matrix(y_test, final_test_pred).ravel()

# Calculate total samples and true class counts in the test set for comprehensive reporting
total_test_samples = len(y_test)
actual_shows_test = (y_test == 0).sum()
actual_noshows_test = (y_test == 1).sum()

# 8. Print the complete final results cleanly and clearly
print("================================================================================")
print("                   FINAL UNBIASED TEST SET EVALUATION                           ")
print("================================================================================")
print(f"Total Test Samples: {total_test_samples}")
print(f"Actual Show (0) cases:    {actual_shows_test}")
print(f"Actual No-show (1) cases: {actual_noshows_test}")
print("--------------------------------------------------------------------------------")
print("FINAL METRICS (Threshold = 0.55):")
print(f"  - Accuracy:  {final_accuracy:.4f}")
print(f"  - Precision: {final_precision:.4f}")
print(f"  - Recall:    {final_recall:.4f}")
print(f"  - F1-Score:  {final_f1:.4f}")
print(f"  - ROC-AUC:   {final_roc_auc:.4f}")
print("--------------------------------------------------------------------------------")
print("FINAL CONFUSION MATRIX BREAKDOWN:")
print(f"  - True Positives (TP - Correctly predicted No-shows): {tp_final}")
print(f"  - False Positives (FP - False Alarms):                {fp_final}")
print(f"  - True Negatives (TN - Correctly predicted Shows):    {tn_final}")
print(f"  - False Negatives (FN - Missed No-shows):             {fn_final}")
print("================================================================================\n")


### Step 26: Feature Audit for the Follow-up Priority System
The goal is to investigate whether the existing MedGuard AI dataset contains legitimate information that could support a patient follow-up priority score. We need to be careful to identify what these features actually represent (e.g. presence of a condition vs. severity of a condition).

In [ ]:
# 1. Create an empty list to hold the structured audit results for every feature in the dataset
feature_audit_data = []

# 2. Loop through every column (feature) in the main dataframe 'df'
for col in df.columns:
    # Determine the fundamental pandas data type of the column
    dtype = str(df[col].dtype)
    
    # Calculate the number of unique values in the column
    unique_count = df[col].nunique()
    
    # Classify the feature as 'Numeric' or 'Categorical/Binary' based on unique counts and dtype
    # Since this is an engineered dataset, most boolean/categorical flags are int64 with 2 unique values
    is_numeric = "Numeric" if (df[col].dtype in ['int64', 'float64'] and unique_count > 2) else "Categorical/Binary"
    
    # Calculate the total number of explicitly missing (NaN) values in the column
    missing_count = df[col].isnull().sum()
    
    # Append the statistics as a dictionary to our audit list
    feature_audit_data.append({
        "Feature": col,
        "Data Type": dtype,
        "Type": is_numeric,
        "Unique Values": unique_count,
        "Missing Values": missing_count
    })

# Convert the audit list into a structured Pandas DataFrame for clean viewing
audit_df = pd.DataFrame(feature_audit_data)

# Print the complete list of dataset features and their statistical properties
print("================================================================================")
print("                           COMPLETE FEATURE AUDIT                               ")
print("================================================================================")
print(audit_df.to_string(index=False))
print("================================================================================\n")

# 3 & 4. Explicitly categorize and analyze the features based on clinical and behavioral relevance
print("================================================================================")
print("                   FEATURE RELEVANCE AND PRIORITY ANALYSIS                      ")
print("================================================================================")

print("A. Features potentially relevant to appointment/no-show BEHAVIOR:")
print("  - Age (Demographic factor affecting reliability)")
print("  - WaitingDays (Direct logistical factor measuring time between scheduling and appointment)")
print("  - SMS_received (Direct intervention factor affecting memory/logistics)")
print("  - Scholarship (Socioeconomic indicator for Brazilian Bolsa Familia welfare program)")
print("  - Historical engineered features like 'Previous_NoShows' or 'NoShow_Rate' (if present)")

print("\nB. Features potentially relevant to patient follow-up PRIORITY:")
print("  - Age (Vulnerability: Elderly or very young patients may require higher priority follow-up)")
print("  - Hypertension (Chronic cardiovascular condition requiring ongoing management)")
print("  - Diabetes (Chronic metabolic condition with severe complications if unmanaged)")
print("  - Alcoholism (Behavioral/chemical dependency condition impacting overall health stability)")
print("  - Handicap (Physical vulnerability indicator)")

print("\nC. Features potentially relevant to MEDICAL/HEALTH context:")
print("  - Hypertension, Diabetes, Alcoholism, Handicap (Direct health indicators)")
print("  - Age (Proxy for general health fragility)")

print("\nD. CRITICAL PROJECT LIMITATION (Features that CANNOT represent disease severity):")
print("  - We DO NOT have 'Disease Severity' (e.g., Stage 4 Cancer vs. Stage 1).")
print("  - We DO NOT have 'Appointment Urgency' (e.g., Routine Checkup vs. Chest Pain).")
print("  - We DO NOT have 'Current Vitals' or 'Lab Results'.")
print("  - The boolean flags (Hypertension: 0 or 1, Diabetes: 0 or 1) only indicate PRESENCE, not SEVERITY.")
print("  - Therefore, we CANNOT legitimately calculate a true 'clinical severity' score based on these data.")
print("  - Any follow-up priority score we build must be clearly labeled as a VULNERABILITY or RISK profile ")
print("    based on the presence of chronic conditions and age, rather than an acute medical emergency score.")
print("================================================================================\n")


### Step 27: DESIGN THE FOLLOW-UP PRIORITY SCORING LOGIC
In this step, we analyze the available training data to design a transparent scoring logic for patient follow-up priority. This priority score will combine the machine-learned probability of missing an appointment (Risk) with the patient's baseline fragility (Vulnerability), explicitly stating that it is not a clinical severity score.

In [ ]:
# Import numpy for numerical operations
import numpy as np

# Print header for the vulnerability indicator distribution
print("================================================================================")
print("             VULNERABILITY INDICATORS: TRAINING DATA DISTRIBUTION               ")
print("================================================================================")

# 1. Show the distribution of Age using summary statistics from the training set
print("1. Age Distribution (X_train):")
print(X_train['Age'].describe().to_string())

# 2. Show the value counts for Hypertension from the training set
print("\n2. Hypertension Value Counts (X_train):")
print(X_train['Hypertension'].value_counts().to_string())

# 3. Show the value counts for Diabetes from the training set
print("\n3. Diabetes Value Counts (X_train):")
print(X_train['Diabetes'].value_counts().to_string())

# 4. Show the value counts for Alcoholism from the training set
print("\n4. Alcoholism Value Counts (X_train):")
print(X_train['Alcoholism'].value_counts().to_string())

# 5. Show the value counts for Handicap from the training set
print("\n5. Handicap Value Counts (X_train):")
print(X_train['Handicap'].value_counts().to_string())
print("================================================================================\n")

# Print header for the formal scoring design proposal
print("================================================================================")
print("             FOLLOW-UP PRIORITY SCORING DESIGN PROPOSAL                         ")
print("================================================================================")

print("A. What the No-show Risk component means:")
print("   - It represents the mathematical probability that a patient will fail to attend")
print("     their scheduled appointment, derived entirely from our locked XGBoost model.")

print("\nB. What the Vulnerability component means:")
print("   - It is a composite indicator of patient fragility and baseline health risks,")
print("     based strictly on age and the presence of tracked chronic conditions.")

print("\nC. Why these are NOT equivalent to disease severity:")
print("   - These flags indicate the presence of a condition (e.g., Diabetes = True), not")
print("     its severity (e.g., well-managed vs. unmanaged).")
print("   - Missing an appointment does not automatically guarantee disease progression;")
print("     it only flags a higher risk of losing continuity of care.")

print("\nD. Which variables could be used:")
print("   - Age (normalized, e.g., using Min-Max scaling to a 0-1 range).")
print("   - Hypertension, Diabetes, Alcoholism, Handicap (binary flags scaled appropriately).")
print("   - final_test_prob (the locked No-show probability).")

print("\nE. Which variables should NOT be used and why:")
print("   - SMS_received, WaitingDays, Scholarship, Neighbourhood: These are logistical")
print("     and socioeconomic variables. While they predict NO-SHOW BEHAVIOR, they do not")
print("     intrinsically alter baseline physical health vulnerability.")

print("\nF. How the two components could eventually be combined:")
print("   - Component 1 (Risk): Use the raw XGBoost probability (0.0 to 1.0).")
print("   - Component 2 (Vulnerability): Create a normalized index (0.0 to 1.0) summing")
print("     scaled age and chronic conditions.")
print("   - Final Priority Score: A combination of these two components (e.g., Risk * Vulnerability)")
print("     to rank patients who are BOTH highly vulnerable AND highly likely to no-show.")

print("\nG. What assumptions would need to be validated:")
print("   - We assume older age linearly or categorically correlates with vulnerability.")
print("   - We assume all available chronic conditions roughly contribute to vulnerability.")
print("   - We assume a multiplicative or additive combination reflects clinical utility.")

print("\n!!! EXPLICIT DISCLAIMER !!!")
print("This Follow-up Priority Score is a decision-support/risk-prioritization feature and is NOT a clinical severity or emergency score.")
print("================================================================================\n")


### Step 28: Build and Inspect the Vulnerability Index
Here we create a prototype transparent vulnerability index using available health indicators. This index aims to quantify patient fragility using Age, Hypertension, Diabetes, Alcoholism, and Handicap. Age is normalized to a 0-1 scale after removing invalid data (-1), and Handicap (which ranges from 0-4) is similarly scaled. The components are summed and averaged to form a transparent Vulnerability Index between 0 and 1.

In [ ]:
# Create a derived dataframe specifically for this calculation to avoid modifying X_train
import numpy as np
vuln_df = X_train[['Age', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap']].copy()

# 1. Handle Invalid Age values (-1)
num_invalid_age = (vuln_df['Age'] == -1).sum()
num_missing_age = vuln_df['Age'].isna().sum()

# Replace invalid age (-1) with NaN so it doesn't skew normalization
vuln_df['Valid_Age'] = vuln_df['Age'].replace(-1, np.nan)

valid_age_min = vuln_df['Valid_Age'].min()
valid_age_max = vuln_df['Valid_Age'].max()

print("================================================================================")
print("             VULNERABILITY INDEX: DATA INSPECTION                               ")
print("================================================================================")
print(f"Valid Age range: {valid_age_min} to {valid_age_max}")
print(f"Number of Age values equal to -1: {num_invalid_age}")
print(f"Number of missing Age values (original): {num_missing_age}")
print("\nHandicap value counts:")
print(vuln_df['Handicap'].value_counts().to_string())

# 2. Create normalized components (0 to 1 range)
# Age normalization (Min-Max scaling using valid min and max)
vuln_df['Age_Component'] = (vuln_df['Valid_Age'] - valid_age_min) / (valid_age_max - valid_age_min)

# Binary health indicators are already 0 or 1, but we map them clearly
vuln_df['Hypertension_Component'] = vuln_df['Hypertension'].astype(float)
vuln_df['Diabetes_Component'] = vuln_df['Diabetes'].astype(float)
vuln_df['Alcoholism_Component'] = vuln_df['Alcoholism'].astype(float)

# Handicap normalization: values are 0, 1, 2, 3, 4, so divide by max possible (4)
vuln_df['Handicap_Component'] = vuln_df['Handicap'] / 4.0

# 3. Calculate Prototype Vulnerability Index
# Using transparent equal weighting across the 5 valid components
component_cols = ['Age_Component', 'Hypertension_Component', 'Diabetes_Component', 
                  'Alcoholism_Component', 'Handicap_Component']

# If a component is NaN (like invalid Age), we do not treat it as 0. 
# We use pandas mean() across rows, which ignores NaNs and calculates the mean of AVAILABLE components.
vuln_df['Vulnerability_Index'] = vuln_df[component_cols].mean(axis=1)

# Track how many rows had missing/unknown information
num_with_missing = vuln_df[component_cols].isna().any(axis=1).sum()

print("\n================================================================================")
print("Prototype Patient Vulnerability Index — NOT a clinical severity score.")
print("================================================================================")
print("\n1. Formula used: Average of available normalized components")
print("   Vulnerability_Index = (Age_Comp + Hyp_Comp + Diab_Comp + Alcoh_Comp + Hand_Comp) / N_valid_components")
print("\n2. Component definitions:")
print("   - Age_Comp: (Age - Min_Age) / (Max_Age - Min_Age) [NaN if Age == -1]")
print("   - Hyp_Comp: Hypertension (0.0 or 1.0)")
print("   - Diab_Comp: Diabetes (0.0 or 1.0)")
print("   - Alcoh_Comp: Alcoholism (0.0 or 1.0)")
print("   - Hand_Comp: Handicap / 4.0 (0.0 to 1.0)")
print("\n3. Component weights:")
print("   - Equal weighting (1.0) across all valid components.")
print("   - If a component is missing, it is omitted from the average, preserving the 0-1 scale.")

print("\n4. Minimum Vulnerability_Index: {:.4f}".format(vuln_df['Vulnerability_Index'].min()))
print("5. Maximum Vulnerability_Index: {:.4f}".format(vuln_df['Vulnerability_Index'].max()))
print("6. Mean Vulnerability_Index:    {:.4f}".format(vuln_df['Vulnerability_Index'].mean()))
print("7. Median Vulnerability_Index:  {:.4f}".format(vuln_df['Vulnerability_Index'].median()))
print(f"8. Number of patients with missing/unknown information: {num_with_missing}")

print("================================================================================")
print("             EXAMPLE ROWS                                                       ")
print("================================================================================")
display_cols = ['Age', 'Hypertension', 'Diabetes', 'Alcoholism', 'Handicap', 'Vulnerability_Index']
print(vuln_df[display_cols].sample(10, random_state=42).to_string())
print("================================================================================\n")


### Step 29: Validate the Vulnerability Index Components
We critically evaluate the components of the prototype Vulnerability Index to ensure transparency and prevent unjustified clinical assumptions.

In [ ]:
# ==============================================================================
# Step 29: Validate the Vulnerability Index Components
# ==============================================================================
# This step critically evaluates the components of the prototype Vulnerability Index
# to ensure transparency and prevent unjustified clinical assumptions.

print("================================================================================")
print("             PART A — HANDICAP CODING                                           ")
print("================================================================================")
# Calculate frequencies and percentages of unique Handicap values
handicap_counts = vuln_df['Handicap'].value_counts()
handicap_pcts = vuln_df['Handicap'].value_counts(normalize=True) * 100

# Combine into a single dataframe for clean printing
handicap_summary = pd.DataFrame({
    'Count': handicap_counts,
    'Percentage (%)': handicap_pcts
})
print("Handicap Value Frequencies:")
print(handicap_summary.to_string())

print("\nHandicap coding cannot be assumed to represent a linear vulnerability scale from the available dataset information.")
print("================================================================================\n")

print("================================================================================")
print("             PART B — COMPONENT CONTRIBUTION                                    ")
print("================================================================================")
# Calculate summary statistics for each individual normalized component
component_stats = vuln_df[component_cols].agg(['mean', 'median', 'min', 'max']).T
print(component_stats.to_string())
print("================================================================================\n")

print("================================================================================")
print("             PART C — DISTRIBUTION                                              ")
print("================================================================================")
# Calculate specified distribution metrics for the Vulnerability Index
v_mean = vuln_df['Vulnerability_Index'].mean()
v_median = vuln_df['Vulnerability_Index'].median()
v_std = vuln_df['Vulnerability_Index'].std()
v_25 = vuln_df['Vulnerability_Index'].quantile(0.25)
v_75 = vuln_df['Vulnerability_Index'].quantile(0.75)
v_90 = vuln_df['Vulnerability_Index'].quantile(0.90)
v_95 = vuln_df['Vulnerability_Index'].quantile(0.95)

print(f"Mean:   {v_mean:.4f}")
print(f"Median: {v_median:.4f}")
print(f"StdDev: {v_std:.4f}")
print(f"25th %: {v_25:.4f}")
print(f"75th %: {v_75:.4f}")
print(f"90th %: {v_90:.4f}")
print(f"95th %: {v_95:.4f}")
print("================================================================================\n")

print("================================================================================")
print("             PART D — CORRELATION CHECK                                         ")
print("================================================================================")
# Calculate correlation between Vulnerability_Index and each component
# This is purely exploratory and does not imply clinical validity
correlations = vuln_df[component_cols].corrwith(vuln_df['Vulnerability_Index'])
print("Correlation with Vulnerability_Index:")
print(correlations.to_string())
print("================================================================================\n")

print("================================================================================")
print("             PART E — IMPORTANT CONCLUSION                                      ")
print("================================================================================")
print("CONCLUSION:")
print("The current Vulnerability_Index should be kept only as a transparent prototype heuristic.")
print("It lacks clinical validation, treats unequal conditions equally, and inappropriately assumes")
print("Handicap values are a linear scale. It must NOT be used as a final clinical score.")
print("================================================================================\n")


### Step 30: DESIGN THE FINAL FOLLOW-UP PRIORITY LOGIC
Based on our validation findings, we design a transparent, two-dimensional Follow-up Priority Logic that rigorously separates operational No-show risk from baseline Patient Vulnerability, avoiding unjustified clinical assumptions.

In [ ]:
# ==============================================================================
# Step 30: DESIGN THE FINAL FOLLOW-UP PRIORITY LOGIC
# ==============================================================================
# Based on our validation findings, we design a transparent, two-dimensional 
# Follow-up Priority Logic that rigorously separates operational No-show risk 
# from baseline Patient Vulnerability, avoiding unjustified clinical assumptions.

print("================================================================================")
print("             VULNERABILITY INDICATOR COUNT DISTRIBUTION (X_train)               ")
print("================================================================================")
# We inspect the distribution of simple indicator counts in the training set
# to help define transparent vulnerability categories without arbitrary medical weights.

# 1. Define simple binary indicators for presence of vulnerability factors
# We treat Handicap > 0 as a simple presence indicator (1), avoiding linear assumptions
# We flag Age >= 65 as a binary indicator of senior vulnerability, avoiding assumptions about exact age values
indicator_df = pd.DataFrame()
indicator_df['Has_Senior_Age'] = (X_train['Age'] >= 65).astype(int)
indicator_df['Has_Hypertension'] = X_train['Hypertension'].astype(int)
indicator_df['Has_Diabetes'] = X_train['Diabetes'].astype(int)
indicator_df['Has_Alcoholism'] = X_train['Alcoholism'].astype(int)
indicator_df['Has_Handicap'] = (X_train['Handicap'] > 0).astype(int)

# 2. Count the total number of present indicators per patient
indicator_df['Total_Indicators'] = indicator_df.sum(axis=1)

# 3. Display the distribution of total indicator counts
indicator_counts = indicator_df['Total_Indicators'].value_counts().sort_index()
indicator_pcts = indicator_df['Total_Indicators'].value_counts(normalize=True).sort_index() * 100

summary_df = pd.DataFrame({
    'Count': indicator_counts,
    'Percentage (%)': indicator_pcts
})

print("Number of Documented Vulnerability Indicators per Patient:")
print(summary_df.to_string())
print("================================================================================\n")


print("================================================================================")
print("             FINAL FOLLOW-UP PRIORITY DESIGN PROPOSAL                           ")
print("================================================================================")

print("A. Why no-show probability and vulnerability should remain separate:")
print("   - Probability is a mathematical prediction of operational behavior (will they attend?).")
print("   - Vulnerability is a baseline profile of patient fragility.")
print("   - Keeping them separate ensures transparency; a user can clearly see WHY a patient")
print("     was flagged (e.g., highly likely to miss, vs. highly vulnerable if they miss).")

print("\nB. Why the system cannot claim to measure disease severity:")
print("   - The dataset lacks acute medical data (vitals, lab results, disease staging).")
print("   - The flags (like Diabetes) only indicate presence, not whether it is managed or acute.")

print("\nC. Why arbitrary medical weights are being avoided:")
print("   - Assigning higher weights to specific conditions without clinical validation is dangerous.")
print("   - A simple count of documented vulnerability factors is transparent and defensible.")

print("\nD. How the two-dimensional priority matrix would work:")
print("   - Dimension 1 (No-Show Risk): Low (<0.40), Moderate (0.40-0.54), High (>=0.55).")
print("   - Dimension 2 (Vulnerability): E.g., Low (0 indicators), Higher (1+ indicators).")
print("\n   PROPOSED 2D PRIORITY MATRIX:")
print("                       LOW VULNERABILITY     HIGHER VULNERABILITY")
print("   LOW NO-SHOW RISK       Low                    Moderate")
print("   MODERATE RISK          Moderate               High")
print("   HIGH NO-SHOW RISK      High                   Highest")

print("\nE. What information is required to genuinely estimate clinical severity:")
print("   - Current vital signs, active symptoms, triage notes, laboratory test results,")
print("     disease staging, and true appointment urgency (e.g., routine vs. acute).")

print("\nF. What needs to be validated before implementation:")
print("   - The chosen cutoffs for vulnerability (e.g., 1+ vs 2+ indicators) must be")
print("     reviewed by clinical stakeholders to ensure they align with clinic resources.")
print("   - The operational utility of the proposed High/Highest priority lists.")

print("================================================================================\n")


### Step 31: IMPLEMENT THE FOLLOW-UP PRIORITY SYSTEM
We operationalize the Follow-up Priority Matrix designed in Step 30 using the untouched final test set (`X_test`) and the final no-show probabilities (`final_test_prob`). This code creates operational categories without claiming clinical severity.

In [ ]:
# ==============================================================================
# Step 31: IMPLEMENT THE FOLLOW-UP PRIORITY SYSTEM
# ==============================================================================
# We operationalize the Follow-up Priority Matrix designed in Step 30 using the 
# untouched final test set (X_test) and the final no-show probabilities (final_test_prob).
# This code creates operational categories without claiming clinical severity.

# Create a copy of X_test to store our new operational priority columns
priority_df = X_test.copy()

# Add the final XGBoost probabilities to the dataframe
priority_df['final_test_prob'] = final_test_prob

# ------------------------------------------------------------------------------
# DIMENSION 1 — NO-SHOW RISK
# ------------------------------------------------------------------------------
# Define transparent operational categories based strictly on probability thresholds
def categorize_risk(prob):
    if prob < 0.40:
        return 'Low'
    elif prob < 0.55:
        return 'Moderate'
    else:
        return 'High'

priority_df['no_show_risk_category'] = priority_df['final_test_prob'].apply(categorize_risk)

# ------------------------------------------------------------------------------
# DIMENSION 2 — PATIENT VULNERABILITY
# ------------------------------------------------------------------------------
# Define transparent vulnerability indicators without arbitrary medical weights

# Helper function to convert Age into a binary vulnerability indicator
# Invalid age (-1) is treated as unknown (0) to prevent false vulnerability flagging
def get_age_vulnerability(age):
    if age == -1:
        return 0
    elif age < 18 or age >= 60:
        return 1
    else:
        return 0

priority_df['Age_Vuln'] = priority_df['Age'].apply(get_age_vulnerability)

# Handicap counts as ONE vulnerability indicator if any handicap exists (Handicap > 0)
priority_df['Handicap_Vuln'] = (priority_df['Handicap'] > 0).astype(int)

# Sum the binary indicators to get the total count (range: 0 to 5)
priority_df['vulnerability_indicator_count'] = (
    priority_df['Age_Vuln'] + 
    priority_df['Hypertension'].astype(int) + 
    priority_df['Diabetes'].astype(int) + 
    priority_df['Alcoholism'].astype(int) + 
    priority_df['Handicap_Vuln']
)

# Define transparent vulnerability categories based purely on indicator count
def categorize_vulnerability(count):
    if count == 0:
        return 'Low Vulnerability'
    else:
        return 'Higher Vulnerability'

priority_df['vulnerability_category'] = priority_df['vulnerability_indicator_count'].apply(categorize_vulnerability)

# ------------------------------------------------------------------------------
# FINAL 2D PRIORITY MATRIX
# ------------------------------------------------------------------------------
# Map the combinations to the final Follow-up Priority
def determine_priority(row):
    risk = row['no_show_risk_category']
    vuln = row['vulnerability_category']
    
    if risk == 'Low':
        if vuln == 'Low Vulnerability':
            return 'Low'
        else: # Higher Vulnerability
            return 'Moderate'
            
    elif risk == 'Moderate':
        if vuln == 'Low Vulnerability':
            return 'Moderate'
        else: # Higher Vulnerability
            return 'High'
            
    elif risk == 'High':
        if vuln == 'Low Vulnerability':
            return 'High'
        else: # Higher Vulnerability
            return 'Highest'

priority_df['follow_up_priority'] = priority_df.apply(determine_priority, axis=1)

# Clean up temporary columns to keep the dataframe concise
priority_df = priority_df.drop(columns=['Age_Vuln', 'Handicap_Vuln'])

# ------------------------------------------------------------------------------
# OUTPUT & VALIDATION
# ------------------------------------------------------------------------------
print("================================================================================")
print("             FOLLOW-UP PRIORITY IMPLEMENTATION SUMMARY (TEST SET)               ")
print("================================================================================")

print("1. No-Show Risk Categories:")
print(priority_df['no_show_risk_category'].value_counts().to_string())

print("\n2. Vulnerability Categories:")
print(priority_df['vulnerability_category'].value_counts().to_string())

print("\n3. Final Follow-Up Priority Categories:")
print(priority_df['follow_up_priority'].value_counts().to_string())

print("\n4. Cross-Tabulation (No-Show Risk vs Vulnerability):")
print(pd.crosstab(priority_df['no_show_risk_category'], priority_df['vulnerability_category']))

print("\n================================================================================")
print("             EXAMPLE PATIENT ROWS                                               ")
print("================================================================================")
display_cols = ['final_test_prob', 'no_show_risk_category', 
                'vulnerability_indicator_count', 'vulnerability_category', 'follow_up_priority']
print(priority_df[display_cols].sample(10, random_state=42).to_string())
print("================================================================================\n")

print("!!! EXPLICIT DISCLAIMER !!!")
print("Follow-up Priority is an operational decision-support classification based on predicted no-show risk and documented vulnerability indicators. It is NOT a clinical severity, diagnosis, or emergency score.")
print("================================================================================\n")


### Step 32: FINAL SYSTEM SANITY CHECK
The purpose of this step is ONLY to verify that the complete MedGuard AI pipeline is logically consistent. No models are retrained or modified.

In [ ]:
# ==============================================================================
# Step 32: FINAL SYSTEM SANITY CHECK
# ==============================================================================
# The purpose of this step is ONLY to verify that the complete MedGuard AI 
# pipeline is logically consistent. No models are retrained or modified.

# Initialize a global flag for tracking check failures
sanity_check_passed = True

print("================================================================================")
print("             CHECK 1 — DATASET CONSISTENCY                                      ")
print("================================================================================")
expected_rows = 22105

# Helper function to print pass/fail for consistency checks
def verify_length(name, obj, expected):
    global sanity_check_passed
    length = len(obj)
    status = "PASS" if length == expected else "FAIL"
    if status == "FAIL": sanity_check_passed = False
    print(f"{name} has {length} rows: {status}")

verify_length("X_test", X_test, expected_rows)
verify_length("y_test", y_test, expected_rows)
verify_length("final_test_prob", priority_df['final_test_prob'], expected_rows)
verify_length("no_show_risk_category", priority_df['no_show_risk_category'], expected_rows)
verify_length("vulnerability_indicator_count", priority_df['vulnerability_indicator_count'], expected_rows)
verify_length("vulnerability_category", priority_df['vulnerability_category'], expected_rows)
verify_length("follow_up_priority", priority_df['follow_up_priority'], expected_rows)


print("\n================================================================================")
print("             CHECK 2 — PROBABILITY VALIDITY                                     ")
print("================================================================================")
min_prob = priority_df['final_test_prob'].min()
max_prob = priority_df['final_test_prob'].max()
missing_prob = priority_df['final_test_prob'].isna().sum()

print(f"Minimum probability: {min_prob}")
print(f"Maximum probability: {max_prob}")
print(f"Missing probability count: {missing_prob}")

if missing_prob > 0 or min_prob < 0 or max_prob > 1:
    print("Probability Validity: FAIL")
    sanity_check_passed = False
else:
    print("Probability Validity: PASS")


print("\n================================================================================")
print("             CHECK 3 — RISK CATEGORY CONSISTENCY                                ")
print("================================================================================")
# Find any rows that violate the risk category logic
inconsistent_risk = priority_df[
    ~(((priority_df['final_test_prob'] < 0.40) & (priority_df['no_show_risk_category'] == 'Low')) |
      ((priority_df['final_test_prob'] >= 0.40) & (priority_df['final_test_prob'] < 0.55) & (priority_df['no_show_risk_category'] == 'Moderate')) |
      ((priority_df['final_test_prob'] >= 0.55) & (priority_df['no_show_risk_category'] == 'High')))
]
inconsistent_risk_count = len(inconsistent_risk)

print(f"Expected inconsistent rows: 0")
print(f"Found inconsistent rows: {inconsistent_risk_count}")
if inconsistent_risk_count > 0:
    print("Risk Category Consistency: FAIL")
    sanity_check_passed = False
else:
    print("Risk Category Consistency: PASS")


print("\n================================================================================")
print("             CHECK 4 — VULNERABILITY CATEGORY CONSISTENCY                       ")
print("================================================================================")
# Find any rows that violate the vulnerability category logic
inconsistent_vuln = priority_df[
    ~(((priority_df['vulnerability_indicator_count'] == 0) & (priority_df['vulnerability_category'] == 'Low Vulnerability')) |
      ((priority_df['vulnerability_indicator_count'] >= 1) & (priority_df['vulnerability_category'] == 'Higher Vulnerability')))
]
inconsistent_vuln_count = len(inconsistent_vuln)

print(f"Expected inconsistent rows: 0")
print(f"Found inconsistent rows: {inconsistent_vuln_count}")
if inconsistent_vuln_count > 0:
    print("Vulnerability Category Consistency: FAIL")
    sanity_check_passed = False
else:
    print("Vulnerability Category Consistency: PASS")


print("\n================================================================================")
print("             CHECK 5 — PRIORITY MATRIX CONSISTENCY                              ")
print("================================================================================")
# Map expected priority combinations
expected_matrix = {
    ('Low', 'Low Vulnerability'): 'Low',
    ('Low', 'Higher Vulnerability'): 'Moderate',
    ('Moderate', 'Low Vulnerability'): 'Moderate',
    ('Moderate', 'Higher Vulnerability'): 'High',
    ('High', 'Low Vulnerability'): 'High',
    ('High', 'Higher Vulnerability'): 'Highest'
}

# Create a temporary column to compare expected vs actual priority
priority_df['expected_priority'] = priority_df.apply(
    lambda row: expected_matrix.get((row['no_show_risk_category'], row['vulnerability_category'])), axis=1
)

inconsistent_priority = priority_df[priority_df['follow_up_priority'] != priority_df['expected_priority']]
inconsistent_priority_count = len(inconsistent_priority)

# Drop the temporary column
priority_df = priority_df.drop(columns=['expected_priority'])

print(f"Expected inconsistent rows: 0")
print(f"Found inconsistent rows: {inconsistent_priority_count}")
if inconsistent_priority_count > 0:
    print("Priority Matrix Consistency: FAIL")
    sanity_check_passed = False
else:
    print("Priority Matrix Consistency: PASS")


print("\n================================================================================")
print("             CHECK 6 — CATEGORY TOTALS                                          ")
print("================================================================================")
risk_total = priority_df['no_show_risk_category'].value_counts().sum()
vuln_total = priority_df['vulnerability_category'].value_counts().sum()
priority_total = priority_df['follow_up_priority'].value_counts().sum()

print(f"No-show risk category counts sum to {risk_total} (Expected: {expected_rows})")
print(f"Vulnerability category counts sum to {vuln_total} (Expected: {expected_rows})")
print(f"Follow-up priority category counts sum to {priority_total} (Expected: {expected_rows})")

if risk_total != expected_rows or vuln_total != expected_rows or priority_total != expected_rows:
    print("Category Totals: FAIL")
    sanity_check_passed = False
else:
    print("Category Totals: PASS")


print("\n================================================================================")
print("             CHECK 7 — FINAL PIPELINE SUMMARY                                   ")
print("================================================================================")
print("Raw patient data")
print("        ↓")
print("Data preprocessing")
print("        ↓")
print("XGBoost prediction")
print("        ↓")
print("No-show probability")
print("        ↓")
print("Operational risk category")
print("        +")
print("Documented vulnerability indicators")
print("        ↓")
print("Follow-up Priority")
print("\nMedGuard AI pipeline sanity check completed.")

print("\n================================================================================")
if sanity_check_passed:
    print("FINAL SYSTEM SANITY CHECK: PASSED")
else:
    print("FINAL SYSTEM SANITY CHECK: FAILED")
print("================================================================================\n")


### Step 33: SAVE THE FINAL MODEL ARTIFACT
We now persist the explicitly locked and trained XGBoost model along with its configuration for future deployment. No additional modifications or evaluations are performed here.

In [ ]:
# ==============================================================================
# Step 33: SAVE THE FINAL MODEL ARTIFACT
# ==============================================================================
# We now persist the exactly locked and trained XGBoost model along with its 
# configuration for future use (e.g., Streamlit deployment) without modifying it.

import joblib
import json
import os

# Define the directory where models will be saved (relative to the notebook folder)
models_dir = "../models"

# Ensure the models directory exists; if not, create it
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

# Define the exact file paths for the artifacts
model_file_path = os.path.join(models_dir, "medguard_xgboost_final.pkl")
config_file_path = os.path.join(models_dir, "medguard_model_config.json")

# ------------------------------------------------------------------------------
# SAVE ARTIFACTS
# ------------------------------------------------------------------------------
# Save the trained XGBoost model using joblib for efficient serialization
joblib.dump(final_xgboost_model, model_file_path)

# Extract the number of features the final model was trained on from X_train
feature_count = X_train.shape[1]

# Define the final locked configuration dictionary exactly as specified
model_config = {
    "model_type": "XGBoost",
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "eval_metric": "logloss",
    "operating_threshold": 0.55,
    "class_imbalance_strategy": "scale_pos_weight",
    "feature_count": feature_count
}

# Save the configuration dictionary to a JSON file
with open(config_file_path, "w") as json_file:
    json.dump(model_config, json_file, indent=4)

# ------------------------------------------------------------------------------
# VERIFICATION CHECKS
# ------------------------------------------------------------------------------
# 1 & 2: Check if files physically exist on disk
model_exists = os.path.exists(model_file_path)
config_exists = os.path.exists(config_file_path)

# 3, 4, 5, 6: Verify reloading and parameter correctness
try:
    # Attempt to reload the model from disk
    loaded_model = joblib.load(model_file_path)
    # Check if the loaded object is indeed an XGBClassifier
    model_reload_pass = type(loaded_model).__name__ == "XGBClassifier"
    
    # Attempt to reload the JSON configuration from disk
    with open(config_file_path, "r") as f:
        loaded_config = json.load(f)
    
    # Check if the loaded JSON has the correct threshold
    config_reload_pass = (loaded_config.get("operating_threshold") == 0.55)
    loaded_threshold = loaded_config.get("operating_threshold")
    
except Exception as e:
    model_reload_pass = False
    config_reload_pass = False
    loaded_threshold = "ERROR"

# Determine overall sanity check status based on all conditions
overall_pass = model_exists and config_exists and model_reload_pass and config_reload_pass

# ------------------------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------------------------
print("============================================================")
print("             MEDGUARD AI MODEL ARTIFACT SAVED               ")
print("============================================================")
print("\nModel file:")
print("models/medguard_xgboost_final.pkl")
print("\nConfiguration file:")
print("models/medguard_model_config.json")

# Print the test results formatted cleanly
model_test_str = "PASS" if model_reload_pass else "FAIL"
config_test_str = "PASS" if config_reload_pass else "FAIL"

print(f"\nModel reload test: {model_test_str}")
print(f"Configuration reload test: {config_test_str}")
print(f"Operating threshold: {loaded_threshold}")

print("\n============================================================")
if overall_pass:
    print("FINAL MODEL ARTIFACT CHECK: PASSED")
else:
    print("FINAL MODEL ARTIFACT CHECK: FAILED")
print("============================================================\n")


### Step 35: SAVE AND VERIFY THE FINAL FEATURE SCHEMA
We explicitly extract and save the exact ordered list of features that our final XGBoost model was trained on. This serves as the source of truth for our future prediction pipeline to ensure raw input data is aligned perfectly.

In [ ]:
# ==============================================================================
# Step 35: SAVE AND VERIFY THE FINAL FEATURE SCHEMA
# ==============================================================================
# In this step we extract, verify, and save the exact ordered list of features 
# that our model was trained on, which is critical for future predictions.

import json
import os

# Define the exact file path where we will save the feature schema
schema_file_path = "../models/medguard_feature_schema.json"

# Extract the ordered feature list directly from the training data columns
feature_list = X_train.columns.tolist()
num_train_features = len(feature_list)
num_test_features = X_test.shape[1]

# Extract the number of features the XGBoost model natively expects
xgb_expected_features = final_xgboost_model.n_features_in_

# ------------------------------------------------------------------------------
# SAVE FEATURE SCHEMA
# ------------------------------------------------------------------------------
# Construct the JSON schema dictionary exactly as requested
schema_dict = {
    "feature_count": num_train_features,
    "features": feature_list
}

# Write the schema dictionary to the JSON file
with open(schema_file_path, "w") as f:
    json.dump(schema_dict, f, indent=4)

# ------------------------------------------------------------------------------
# VERIFICATION CHECKS
# ------------------------------------------------------------------------------
# 1. Check if X_train columns exactly match X_test columns (in order)
order_match = list(X_train.columns) == list(X_test.columns)

# 2. Reload the JSON file to verify it saved correctly
with open(schema_file_path, "r") as f:
    reloaded_schema = json.load(f)

# 3. Check if file physically exists
file_exists = os.path.exists(schema_file_path)

# 4. Check if the reloaded feature list exactly equals the original list
reload_match = (reloaded_schema.get("features") == feature_list)

# 5. Check if the reloaded feature count matches X_train.shape[1]
count_match = (reloaded_schema.get("feature_count") == num_train_features)

# 6. Check if XGBoost expects the same number of features
xgb_match = (xgb_expected_features == num_train_features)

# Determine overall sanity check status
overall_pass = order_match and file_exists and reload_match and count_match and xgb_match

# ------------------------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------------------------
print("============================================================")
print("             MEDGUARD AI FEATURE SCHEMA                     ")
print("============================================================")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

print(f"\nNumber of training features: {num_train_features}")
print(f"Number of test features: {num_test_features}")
print(f"XGBoost expected features: {xgb_expected_features}")

print(f"\nTraining/Test feature order identical: {'PASS' if order_match else 'FAIL'}")

print(f"\nFeature schema saved:\nmodels/medguard_feature_schema.json")
print(f"\nFeature schema reload test: {'PASS' if reload_match else 'FAIL'}")
print(f"Feature count verification: {'PASS' if count_match else 'FAIL'}")
print(f"XGBoost feature compatibility: {'PASS' if xgb_match else 'FAIL'}")

print("\n============================================================")
if overall_pass:
    print("FEATURE SCHEMA CHECK: PASSED")
else:
    print("FEATURE SCHEMA CHECK: FAILED")
print("============================================================\n")

# Print the complete feature list with index numbers
print("Complete Feature List:")
for i, feature in enumerate(feature_list):
    print(f"{i}: {feature}")
